# Ground Truth : run final (à lancer entre le 14 nov. 18 h et le 15 nov. 23 h 59)

Ce notebook exécute **exactement** les commandes officielles, depuis le tag `final` de ton dépôt :
`setup.sh`, puis `run.sh index`, `run.sh update` et deux fois `run.sh answer`. Durée : environ 2 h.

**Avant de commencer :** Exécution → Modifier le type d'exécution → **GPU T4** ; PC branché, sans veille ;
ne lance **pas** d'autre notebook GPU en même temps (le journal du générateur doit rester cohérent).

Après chaque étape longue, une cellule sauvegarde sur ton Drive (`gt_finale/`). Si la session coupe,
relance la section 0, puis la cellule « Reprise », et continue à l'étape suivante.

## 0. Installation depuis le tag `final`

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO = "https://github.com/YanisYounsi-code/Ground-Truth-un-RAG-qui-ne-ment-pas.git"
!rm -rf /content/gt && git clone -q --branch final $REPO /content/gt
%cd /content/gt
!git describe --tags && git log --oneline -1

In [ ]:
!bash setup.sh
!python corpus/download_corpus.py
!ls corpus/pdf | wc -l; ls corpus/pdf_v2 | wc -l

In [ ]:
# Questions publiées par le mentor le 14 nov. à 18 h (hors du tag : on les télécharge à part)
!mkdir -p final
!wget -q -O data/test_questions.json https://raw.githubusercontent.com/Mathos34/ground-truth-kit/main/data/test_questions.json
!wget -q -O data/redteam_questions.json https://raw.githubusercontent.com/Mathos34/ground-truth-kit/main/data/redteam_questions.json
!python -c "import json; [print(f, len(json.load(open(f))['questions'] if isinstance(json.load(open(f)), dict) else json.load(open(f)))) for f in ('data/test_questions.json', 'data/redteam_questions.json')]"
# Si le téléchargement échoue : glisse les deux fichiers dans le dossier data/ (panneau Fichiers, à gauche)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/gt_finale

**Reprise (seulement si la session a coupé)** : remet l'index, `final/` et le journal du générateur
sauvegardés, puis reprends à l'étape qui n'était pas terminée.

In [ ]:
!cp -r /content/drive/MyDrive/gt_finale/index /tmp/ 2>/dev/null; ls /tmp/index
!mkdir -p final runs && cp /content/drive/MyDrive/gt_finale/final/* final/ 2>/dev/null; cp /content/drive/MyDrive/gt_finale/generator_log.jsonl runs/ 2>/dev/null; ls final runs

## 1. Index du corpus v1 (limite : 2 h)

In [ ]:
!time bash run.sh index corpus/pdf /tmp/index
!rm -rf /content/drive/MyDrive/gt_finale/index && cp -r /tmp/index /content/drive/MyDrive/gt_finale/index && echo "index sauvegardé"

## 2. Mise à jour avec les 20 PDF v2 (limite : 20 min)

In [ ]:
!time bash run.sh update /tmp/index corpus/pdf_v2
!rm -rf /content/drive/MyDrive/gt_finale/index && cp -r /tmp/index /content/drive/MyDrive/gt_finale/index && echo "index v1 + v2 sauvegardé"

## 3. Test caché (283 questions, limite : 30 s par question en moyenne)

In [ ]:
!time bash run.sh answer /tmp/index data/test_questions.json final/predictions_test.json
!mkdir -p /content/drive/MyDrive/gt_finale/final && cp final/predictions_test.json /content/drive/MyDrive/gt_finale/final/ && cp runs/generator_log.jsonl /content/drive/MyDrive/gt_finale/

## 4. Questions red team de tous les participants

In [ ]:
!time bash run.sh answer /tmp/index data/redteam_questions.json final/predictions_redteam.json
!cp final/predictions_redteam.json /content/drive/MyDrive/gt_finale/final/ && cp runs/generator_log.jsonl /content/drive/MyDrive/gt_finale/

## 5. Journal du générateur et rapatriement

In [ ]:
!cp runs/generator_log.jsonl final/generator_log.jsonl && cp final/* /content/drive/MyDrive/gt_finale/final/
!python -c "import json; [print(f, len(json.load(open(f))['predictions'])) for f in ('final/predictions_test.json', 'final/predictions_redteam.json')]"
!wc -l final/generator_log.jsonl
!cd /content/gt && zip -q -r /content/final.zip final && unzip -l /content/final.zip
from google.colab import files; files.download("/content/final.zip")

Ensuite, sur ton PC : décompresse `final.zip` dans `ground-truth-kit/` (il crée le dossier `final/`), puis
`git add final/ && git commit -m "run final" && git push`. **Ne déplace pas le tag `final`** : seul le
dossier `final/` est ajouté après lui.